# Module 09: EMC in Aircraft
**AE2.021 · USTH**

This notebook recomputes every dB, dBm and cascade number in the lecture (slides 13 to 21), catches the 10 kW versus 100 kW error on slide 21, verifies the square-wave Fourier series, the real capacitor (ESL, ESR, f_r), Tooley's spectrum examples, Floyd's noise margins and Tooley's chapter 14 key (read from the scanned image A.14, p.381).

## 1. dB and dBm

In [1]:
import math
L10=math.log10
dbP=lambda p2,p1:10*L10(p2/p1); dbA=lambda a2,a1:20*L10(a2/a1)
dbm=lambda w:10*L10(w/1e-3); from_dbm=lambda d:1e-3*10**(d/10)

# slide 13: three examples
assert round(dbA(1,1e-3))==60 and round(dbP(1,1e-3))==30 and round(dbA(1e-3,1e-5))==40 and round(dbA(0.1,1))==-20
# slide 14-15: ratio table
for r,pw,vo in ((0.01,-20,-40),(0.1,-10,-20),(0.5,-3,-6),(1,0,0),(2,3,6),(10,10,20),(100,20,40)):
    assert abs(dbP(r,1)-pw)<0.4 and abs(dbA(r,1)-vo)<0.8, r
# slide 17: exercise
rows=[(dbA(200,100),6),(dbA(300,100),10),(dbA(400,100),12),(dbA(500,100),14),(dbP(2,1),3),(dbP(3,1),5),(dbP(5,1),7)]
for got,slide in rows: print(round(got,2),slide); assert abs(got-slide)<0.5
# slide 19: dBm table
for w,d in ((0.001,0),(0.1,20),(0.5,27),(1,30),(2,33),(10,40),(1000,60)):
    print(w,round(dbm(w),2)); assert abs(dbm(w)-d)<0.1
assert from_dbm(0)==1e-3          # 0 dBm = 1 mW, not 0 W

6.02 6
9.54 10
12.04 12
13.98 14
3.01 3
4.77 5
6.99 7
0.001 0.0
0.1 20.0
0.5 26.99
1 30.0
2 33.01
10 40.0
1000 60.0


## 2. Exercises on slides 20 and 21

In [2]:
# slide 20: cascade
assert 0+10+6==16 and 0-3-10==-13
# slide 21
v1=1*10**(-2/20);          print("1) Vout =",round(v1,3)); assert abs(v1-0.794)<1e-3
p2=10*10**(40/10);         print("2) Pout =",p2,"W");      assert p2==100000          # the slide misprints '10000 W (10 kW)'
p3=-20+30;                 print("3) Pout =",p3,"dBm =",from_dbm(p3)*1e3,"mW"); assert p3==10 and abs(from_dbm(p3)-0.01)<1e-12
p4=34+(-2);                print("4) Iout =",p4,"dBuA"); assert p4==32
print("Error on slide 21: the printed result 10000 W differs from the 100000 W of its own calculation.")

1) Vout = 0.794
2) Pout = 100000.0 W
3) Pout = 10 dBm = 10.0 mW
4) Iout = 32 dBuA
Error on slide 21: the printed result 10000 W differs from the 100000 W of its own calculation.


## 3. Square wave and spectrum

In [3]:
import numpy as np
t=np.linspace(0,1,20001,endpoint=False)
def sq(N): return sum(np.sin(2*np.pi*n*t)/n for n in range(1,N+1,2))
# Tooley TYU 14.1: 50 V, 2 kHz -> fifth harmonic
assert 50/5==10 and 5*2000==10000
# The first harmonic (n=1) of the V/n series has amplitude 4/pi * V_square: check the value on the plateau
plateau=sq(401)[len(t)//4]          # middle of the positive half-cycle
print("plateau value of the V/n series =",round(plateau,4),"(pi/4 =",round(math.pi/4,4),")")
assert abs(plateau-math.pi/4)<0.01   # => the real square-wave amplitude = (pi/4) * V, meaning the fundamental = 4V_sq/pi
# Tooley figure 14.9: the third harmonic is 70 dB below the fundamental
assert -20-(-90)==70

plateau value of the V/n series = 0.7866 (pi/4 = 0.7854 )


## 4. A real capacitor

In [4]:
fr=lambda L,C:1/(2*math.pi*math.sqrt(L*C))
Z=lambda f,C,L,R:math.sqrt(R**2+(2*math.pi*f*L-1/(2*math.pi*f*C))**2)
C,Lh,R=100e-9,10e-9,0.05
f0=fr(Lh,C); print("f_r =",round(f0/1e6,3),"MHz"); assert abs(f0-5.033e6)<1e3
assert abs(Z(f0,C,Lh,R)-R)<1e-9                      # tai f_r: Z = ESR
assert Z(0.1*f0,C,Lh,R)>Z(f0,C,Lh,R) and Z(10*f0,C,Lh,R)>Z(f0,C,Lh,R)
# above f_r: the inductive reactance of the inductance dominates
f=10*f0; assert 2*math.pi*f*Lh>1/(2*math.pi*f*C)
# the 1 nF capacitor has a much higher f_r
assert fr(Lh,1e-9)>3*f0

f_r = 5.033 MHz


## 5. Tooley and Floyd numbers

In [5]:
# Tooley figure 14.14: the noise floor rises from -110 to -84 dB = 26 dB; the book writes '50 times'
d=-84-(-110); print("rise",d,"dB ->",round(10**(d/20),1),"times in amplitude,",round(10**(d/10)),"times in power")
assert d==26 and abs(10**(d/20)-19.95)<0.1 and round(10**(d/10))==398    # not 50 times
# Tooley 14.3: spectrum analysis: marker amplitude -26 dBm at 121 MHz, noise floor -95 dBm => 69 dB above the noise floor
assert -26-(-95)==69
# CISPR 11 (slide 26): convert dBuV/m to uV/m
print(round(10**(30/20),1),"uV/m;",round(10**(37/20),1),"uV/m;",round(10**(7/20),2),"lan")
assert abs(10**(37/20)/10**(30/20)-2.239)<0.01
# Floyd Ch.15 Problem 3 and example 15-1 (CMOS 5 V)
VNH=2.4-2.25; VNL=0.65-0.4; assert abs(VNH-0.15)<1e-9 and abs(VNL-0.25)<1e-9
assert abs((4.4-3.5)-0.9)<1e-9 and abs((1.5-0.33)-1.17)<1e-9

rise 26 dB -> 20.0 times in amplitude, 398 times in power
31.6 uV/m; 70.8 uV/m; 2.24 lan


## 6. Tooley Ch.14 key

In [6]:
KEY=dict(zip(range(1,10),"a c c c c a b a c".split()))     # A.14 tr.381
assert KEY[8]=="a"          # power supply: low-pass filter
assert KEY[9]=="c"          # fast-switching and large currents
assert KEY[3]=="c"          # amplitude versus frequency
assert KEY[6]=="a"          # < 0.005 ohm (Tooley 14.6 says < 0.001-0.003 ohm)
# Bonding thresholds: slide 43 (0.001-0.003), slide 45 (<2.5 mohm, tester 0.003 ohm), Tooley Q6 (<0.005)
tester=0.003; slide45_limit=0.0025
print("tester",tester,"ohm  > limit of slide 45",slide45_limit,"ohm ->", "FAILS by the slide itself" if tester>slide45_limit else "passes")
assert tester>slide45_limit and tester<0.005
print("Key 9/9 read from the scan image.")

tester 0.003 ohm  > limit of slide 45 0.0025 ohm -> FAILS by the slide itself
Key 9/9 read from the scan image.


#### 📤 Takeaway
Every dB figure agrees, except the 10 kW result box on slide 21 (it should be 100 kW). In the V/n series V must be read as the fundamental amplitude (the real square-wave amplitude is π/4 times the series sum). A 26 dB rise in the noise floor is about 20 times in amplitude, not 50. The two bonding resistance thresholds (slides 43 and 45) conflict and the tester picture on slide 45 itself fails.